<div dir="rtl" align="right">
<h2>مقایسهٔ Single-Task و Multi-Task در BUSI</h2>
<p>هدف: آیا آموزش هم‌زمان طبقه‌بندی و سگمنتیشن، نسبت به آموزش جداگانهٔ هرکدام، بهتر عمل می‌کند؟ این نوت‌بوک مستقل است و به اجرای سلول‌های نوت‌بوک‌های قبلی نیاز ندارد. فایل‌های دیتاست و تقسیم‌بندی قبلی باید در Drive باقی باشند.</p>
<p>GPU را از مسیر <span dir="ltr">Runtime → Change runtime type → GPU</span> فعال کن، سپس <span dir="ltr">Run all</span> را بزن و Drive را وصل کن.</p>
<p>سه حالت اجرا می‌شوند: <code dir="ltr">classification_only</code>، <code dir="ltr">segmentation_only</code> و <code dir="ltr">multitask</code>. هرکدام با seedهای 42، 43 و 44: مجموعاً <b>۹ آموزش، هرکدام ۳۰ epoch</b>. هر حالت فقط هدف‌های مربوط به وظیفهٔ خودش را دریافت می‌کند.</p>
<p>برای مقایسهٔ منصفانه، در هر seed بخش مشترک و سرِ وظیفهٔ متناظر دقیقاً وزن اولیهٔ یکسان دارند، اما مدل‌ها جداگانه بارگذاری می‌شوند. فقط MobileNetV2 از ImageNet وزن می‌گیرد؛ Transformer و سرهای پیش‌بینی از ابتدا ساخته می‌شوند. از وزن‌های آموزش‌دیدهٔ BUSI قبلی استفاده نمی‌شود تا مدل تک‌وظیفه‌ای اطلاعاتی از وظیفهٔ دیگر به ارث نبرد.</p>
<p>این آزمایش یک مقایسهٔ کنترل‌شده روی هستهٔ CNN + Transformer و سرهای مستقل است. CNN ثابت است و Transformer آموزش می‌بیند. LPRM، وزن‌دهی هزینه‌ای کلاس‌ها و وزن‌دهی تطبیقی وظایف در این مقایسه غیرفعال‌اند؛ lossها با وزن ثابت یک استفاده می‌شوند. این تنظیمات در هر سه حالت یکسان است. نتیجه، ارزیابی دوبارهٔ مدل کامل قبلی نیست.</p>
<p>تقسیم‌بندی قبلی حفظ می‌شود: ۴۹۹ تصویر train و ۱۱۵ تصویر validation. تصویر و ماسک باهم horizontal flip می‌شوند؛ ترتیب داده‌ها و flipها برای هر seed و epoch بین حالت‌ها یکسان است. هیچ تصویر test خوانده نمی‌شود.</p>
</div>


In [ ]:
#@title ۱. اتصال به Drive و بررسی GPU
from pathlib import Path
import tensorflow as tf
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("ابتدا GPU را از Runtime → Change runtime type فعال کن و دوباره Run all را بزن.")


In [ ]:
#@title ۲. تعریف تمام توابع مستقل
from pathlib import Path


import hashlib


import json


import math


import os


import platform


import numpy as np


import pandas as pd


from PIL import Image


import tensorflow as tf


@tf.keras.utils.register_keras_serializable(package="BUSI_MTL")
class SpatialPositionEmbedding(tf.keras.layers.Layer):
    def build(self, input_shape):
        self.positions = self.add_weight(
            name="positions", shape=(1, int(input_shape[1]), int(input_shape[2])),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02), trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        return inputs + self.positions


def soft_dice_parts(y_true, y_pred):
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    true_area = tf.reduce_sum(y_true, axis=(1, 2, 3))
    predicted_area = tf.reduce_sum(y_pred, axis=(1, 2, 3))
    intersection = tf.reduce_sum(y_true*y_pred, axis=(1, 2, 3))
    dice = (2*intersection+1e-6)/(true_area+predicted_area+1e-6)
    return dice, tf.cast(true_area > 0, tf.float32)


def segmentation_loss_per_image(y_true, y_pred):
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    bce = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_true, y_pred), axis=(1, 2))
    dice, positive = soft_dice_parts(y_true, y_pred)
    return bce + positive*(1-dice)


def _write_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf8")
    os.replace(tmp, path)


def _save_model(model, path):
    path = Path(path)
    tmp = path.with_name(path.stem+".tmp.keras")
    model.save(str(tmp))
    os.replace(tmp, path)


def _read_data(proposed_dir, manifest, root, data):
    frames = {split: manifest[manifest["split"] == split].reset_index(drop=True)
              for split in ["train", "val"]}
    labels = {"normal": 0, "benign": 1, "malignant": 2}
    source = json.loads((proposed_dir/"source_config.json").read_text())
    stats = source["normalization"]
    mean, std = np.array(stats["mean"], np.float32), np.array(stats["std"], np.float32)
    if data is None:
        print("Restoring only train/validation images and masks; no previous training is rerun.")
        data = {}
        for split, frame in frames.items():
            images = np.empty((len(frame), 224, 224, 3), np.float32)
            masks = np.empty((len(frame), 224, 224, 1), np.float32)
            for i, row in frame.iterrows():
                with Image.open(root/row["image"]) as image:
                    images[i] = np.asarray(image.convert("RGB").resize((224,224), Image.Resampling.BILINEAR))
                with Image.open(root/row["mask"]) as image:
                    masks[i,...,0] = np.asarray(image.convert("L").resize((224,224), Image.Resampling.NEAREST)) > 0
                if i % 100 == 0:
                    print(split, i+1, "/", len(frame))
            data["X_"+split] = (images-mean)/std
            data["M_"+split] = masks
            data["y_"+split] = frame["class_name"].map(labels).to_numpy(dtype=np.int32)
    else:
        data = dict(data)
        print("Using the train/validation arrays already in memory.")
    for split, frame in frames.items():
        n = len(frame)
        assert data["X_"+split].shape == (n,224,224,3)
        assert data["M_"+split].shape == (n,224,224,1)
        assert np.array_equal(data["y_"+split], frame["class_name"].map(labels).to_numpy())
        assert np.isfinite(data["X_"+split]).all()
        assert np.isin(data["M_"+split], [0,1]).all()
        assert np.array_equal(data["M_"+split].sum((1,2,3)) > 0, data["y_"+split] != 0)
    assert np.allclose(data["X_train"].mean((0,1,2), dtype=np.float64), 0, atol=0.01), "Expected training Z-score inputs."
    return data, frames


import gc


import shutil


import tempfile


import matplotlib.pyplot as plt


def _repair_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _repair_guard(path, value):
    if path.is_file():
        if json.loads(path.read_text()) != value:
            raise ValueError("این پوشه مربوط به تنظیمات دیگری است؛ خروجی قبلی را حفظ کن: "+str(path))
    else:
        _write_json(path, value)


def _repair_choose_protocol(experiments_root, evaluation_dir):
    if evaluation_dir is not None:
        return Path(evaluation_dir)
    found = sorted(Path(experiments_root).glob("proposed_model_*/ablations/component_study_v1/final_evaluation_v1/frozen_protocol.json"))
    if not found:
        raise FileNotFoundError("پروژهٔ ارزیابی نهایی قبلی پیدا نشد: "+str(experiments_root))
    for i,path in enumerate(found):
        print(f"[{i}] {path.parent}")
    answer = "0" if len(found)==1 else input("شمارهٔ پروژهٔ قبلی را وارد کن: ").strip()
    if not answer.isdigit() or not 0<=int(answer)<len(found):
        raise ValueError("شمارهٔ پروژه معتبر نیست")
    return found[int(answer)].parent


def _phase_leaves(model,prefix=""):
    found=[]
    for layer in model.layers:
        path=prefix+layer.name
        if isinstance(layer,tf.keras.Model):
            found.extend(_phase_leaves(layer,path+"/"))
        else:
            found.append((path,layer))
    return found


def _phase_frozen_digest(network,allowed):
    digest=hashlib.sha256()
    for path,layer in _phase_leaves(network.get_layer("shared_encoder")):
        if path in allowed:
            continue
        for weight in layer.weights:
            array=np.ascontiguousarray(weight.numpy())
            digest.update(str((path,weight.name,array.shape,array.dtype.str)).encode())
            digest.update(memoryview(array).cast("B"))
    return digest.hexdigest()


def _phase_epoch_arrays(data,seed,epoch):
    """Identical shuffling and flips for both conditions in the same seed/epoch."""
    n=len(data["y_train"])
    order=np.random.default_rng(np.random.SeedSequence([seed,epoch,4801])).permutation(n)
    flags=np.random.default_rng(np.random.SeedSequence([seed,epoch,4802])).random(n)<.5
    x,m=data["X_train"][order].copy(),data["M_train"][order].copy()
    x[flags]=x[flags,:,::-1,:]
    m[flags]=m[flags,:,::-1,:]
    return x,m,data["y_train"][order]


from sklearn.metrics import roc_auc_score


"""Fresh, paired single-task versus multi-task CORE study; validation only."""
STMTL_VERSION = "1.0.0"
STMTL_SEEDS = [42, 43, 44]
STMTL_MODES = {
    "classification_only": ("classification",),
    "segmentation_only": ("segmentation",),
    "multitask": ("classification", "segmentation"),
}
STMTL_NAMES = ["normal", "benign", "malignant"]
STMTL_EPOCHS = 30
STMTL_BATCH = 8


@tf.keras.utils.register_keras_serializable(package="BUSI_STMTL")
def stmtl_segmentation_loss(y_true, y_pred):
    # Keras reduces this per-image vector over the batch, including empty masks.
    return segmentation_loss_per_image(y_true, y_pred)


def _stmtl_weight_digest(model):
    digest = hashlib.sha256()
    for index, variable in enumerate(model.weights):
        array = np.ascontiguousarray(variable.numpy())
        digest.update(str((index, array.shape, array.dtype.str)).encode())
        digest.update(memoryview(array).cast("B"))
    return digest.hexdigest()


def _stmtl_build_joint(seed):
    """CNN has ImageNet weights; Transformer and both heads are freshly seeded."""
    tf.keras.utils.set_random_seed(seed)
    L = tf.keras.layers
    base = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3), include_top=False, weights="imagenet", alpha=1.0)
    endpoints = ["block_1_expand_relu", "block_3_expand_relu", "block_6_expand_relu",
                 "block_13_expand_relu", "out_relu"]
    encoder = tf.keras.Model(base.input, [base.get_layer(n).output for n in endpoints],
                             name="shared_encoder")
    expected = [(112,112,96), (56,56,144), (28,28,192), (14,14,576), (7,7,1280)]
    if [tuple(t.shape[1:]) for t in encoder.outputs] != expected:
        raise ValueError("Unexpected MobileNetV2 endpoint shapes")
    encoder.trainable = False
    image = L.Input((224,224,3), name="raw_rgb_image")
    prepared = L.Rescaling(1/127.5, offset=-1, name="imagenet_preprocessing")(image)
    features = encoder(prepared, training=False)
    z = L.Conv2D(128, 1, name="token_projection")(features[-1])
    z = L.Reshape((49,128), name="spatial_tokens")(z)
    z = SpatialPositionEmbedding(name="spatial_positions")(z)
    q = L.LayerNormalization(epsilon=1e-6, name="attention_norm")(z)
    q = L.MultiHeadAttention(num_heads=4, key_dim=32, dropout=.1,
                             name="spatial_self_attention")(q,q)
    z = L.Add(name="attention_residual")([z,q])
    q = L.LayerNormalization(epsilon=1e-6, name="feedforward_norm")(z)
    q = L.Dense(256, activation="gelu", name="feedforward_expand")(q)
    q = L.Dropout(.1, name="feedforward_dropout")(q)
    q = L.Dense(128, name="feedforward_project")(q)
    z = L.Add(name="feedforward_residual")([z,q])
    z = L.Reshape((7,7,128), name="spatial_map")(z)
    z = L.Conv2D(1280, 1, kernel_initializer="zeros", bias_initializer="zeros",
                 name="hybrid_projection")(z)
    hybrid = L.Add(name="hybrid_shared_features")([features[-1],z])
    shared = tf.keras.Model(image, list(features[:-1])+[hybrid], name="shared_representation")
    # Never set shared.trainable=True here: it would recursively unfreeze its CNN.
    if encoder.trainable or not shared.trainable_variables:
        raise RuntimeError("The Transformer must be trainable and the CNN frozen")

    class_input = L.Input((7,7,1280), name="class_feature_input")
    c = L.GlobalAveragePooling2D(name="class_pool")(class_input)
    c = L.Dense(128, activation="relu", name="class_hidden")(c)
    c = L.Dropout(.3, name="class_dropout")(c)
    c = L.Dense(3, activation="softmax", name="classification")(c)
    class_head = tf.keras.Model(class_input, c, name="classification_head")

    seg_inputs = [L.Input(shape, name=f"seg_feature_{i}") for i,shape in enumerate(expected)]
    s = seg_inputs[-1]
    for i,(filters,gate_width,skip) in enumerate(zip(
            [128,64,32,16], [32,32,16,8], reversed(seg_inputs[:-1])),1):
        s = L.UpSampling2D(2, interpolation="bilinear", name=f"decode_{i}_up")(s)
        a = L.Conv2D(gate_width,1,name=f"gate_{i}_skip")(skip)
        b = L.Conv2D(gate_width,1,name=f"gate_{i}_decode")(s)
        gate = L.Add(name=f"gate_{i}_add")([a,b])
        gate = L.Activation("relu",name=f"gate_{i}_relu")(gate)
        gate = L.Conv2D(1,1,activation="sigmoid",name=f"gate_{i}_weight")(gate)
        filtered = L.Multiply(name=f"gate_{i}_filtered_skip")([skip,gate])
        s = L.Concatenate(name=f"decode_{i}_concat")([s,filtered])
        s = L.SeparableConv2D(filters,3,padding="same",activation="relu",
                              name=f"decode_{i}_conv1")(s)
        s = L.SeparableConv2D(filters,3,padding="same",activation="relu",
                              name=f"decode_{i}_conv2")(s)
    s = L.UpSampling2D(2,interpolation="bilinear",name="mask_up")(s)
    s = L.Conv2D(1,1,activation="sigmoid",name="segmentation")(s)
    seg_head = tf.keras.Model(seg_inputs,s,name="segmentation_head")
    raw = L.Input((224,224,3),name="image")
    f = shared(raw)
    return tf.keras.Model(raw, {"classification":class_head(f[-1]),
                                "segmentation":seg_head(f)}, name="joint_core")


def _stmtl_compile(network, mode):
    active = STMTL_MODES[mode]
    losses = {}
    metrics = {}
    if "classification" in active:
        losses["classification"] = tf.keras.losses.SparseCategoricalCrossentropy()
        metrics["classification"] = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
    if "segmentation" in active:
        losses["segmentation"] = stmtl_segmentation_loss
    network.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4,clipnorm=1.0),
                    loss=losses, loss_weights={key:1.0 for key in active}, metrics=metrics)
    return network


def _stmtl_new_view(template_path, mode, seed):
    tf.keras.utils.set_random_seed(seed)
    joint = tf.keras.models.load_model(str(template_path),compile=False)
    outputs = {key:joint.output[key] for key in STMTL_MODES[mode]}
    network = tf.keras.Model(joint.inputs[0],outputs,name=mode)
    shared = network.get_layer("shared_representation")
    encoder = shared.get_layer("shared_encoder")
    if encoder.trainable or encoder.trainable_variables or not shared.trainable_variables:
        raise RuntimeError("Unexpected shared/encoder trainability")
    present = {layer.name for layer in network.layers}
    if mode == "classification_only" and "segmentation_head" in present:
        raise RuntimeError("The classification-only graph still contains the mask head")
    if mode == "segmentation_only" and "classification_head" in present:
        raise RuntimeError("The segmentation-only graph still contains the class head")
    return _stmtl_compile(network,mode)


def _stmtl_targets(mode, labels, masks):
    all_targets = {"classification":labels,"segmentation":masks}
    return {key:all_targets[key] for key in STMTL_MODES[mode]}


def _stmtl_predict(network, images):
    values = {key:[] for key in network.output}
    for start in range(0,len(images),STMTL_BATCH):
        result = network(images[start:start+STMTL_BATCH],training=False)
        if not isinstance(result,dict) or set(result) != set(values):
            raise ValueError("Prediction keys differ from the active task graph")
        for key in values:
            values[key].append(result[key].numpy())
    return {key:np.concatenate(parts,axis=0) for key,parts in values.items()}


def _stmtl_statistics(labels, masks, values):
    """Task-specific metrics: an absent task is absent, never a fabricated zero."""
    y = np.asarray(labels,np.int32)
    n = len(y)
    if not n or not np.isin(y,[0,1,2]).all() or set(values)-{"classification","segmentation"}:
        raise ValueError("Invalid labels or output keys")
    stats, detail = {}, {}
    if "classification" in values:
        p = np.asarray(values["classification"],np.float64)
        if p.shape != (n,3) or not np.isfinite(p).all() or ((p<0)|(p>1)).any():
            raise ValueError("Invalid class probabilities")
        np.testing.assert_allclose(p.sum(1),1,atol=1e-5,rtol=0)
        p = p/p.sum(1,keepdims=True)
        pred = p.argmax(1)
        cm = np.zeros((3,3),np.int64)
        np.add.at(cm,(y,pred),1)
        tp = cm.diagonal().astype(float)
        precision = np.divide(tp,cm.sum(0),out=np.zeros(3),where=cm.sum(0)>0)
        recall = np.divide(tp,cm.sum(1),out=np.zeros(3),where=cm.sum(1)>0)
        f1 = np.divide(2*precision*recall,precision+recall,out=np.zeros(3),where=precision+recall>0)
        malignant = y==2
        if not malignant.any() or malignant.all() or not all((y==i).any() for i in range(3)):
            raise ValueError("Validation must contain all three classes")
        fp = int(((pred==2)&~malignant).sum())
        specificity = 1-fp/int((~malignant).sum())
        confidence,correct = p.max(1),pred==y
        bins = np.minimum(np.searchsorted(np.linspace(0,1,16),confidence,side="right")-1,14)
        ece = sum(float((bins==i).mean())*abs(float(correct[bins==i].mean())-
                    float(confidence[bins==i].mean())) for i in range(15) if (bins==i).any())
        stats.update({"val_accuracy":float(correct.mean()),"val_macro_f1":float(f1.mean()),
            "val_malignant_recall":float(recall[2]),"val_missed_malignant":int((pred[malignant]!=2).sum()),
            "val_nonmalignant_predicted_malignant":fp,"val_nonmalignant_specificity":float(specificity),
            "val_malignant_balanced_accuracy":float((recall[2]+specificity)/2),
            "val_normal_recall":float(recall[0]),
            "val_malignant_auc":float(roc_auc_score(malignant.astype(int),p[:,2])),
            "val_macro_auc_ovr":float(roc_auc_score(y,p,multi_class="ovr",average="macro",labels=[0,1,2])),
            "val_ece_15bins":float(ece),
            "val_brier_multiclass_sum":float(((p-np.eye(3)[y])**2).sum(1).mean()),
            "val_negative_log_likelihood":float(-np.log(np.clip(p[np.arange(n),y],1e-7,1)).mean())})
        detail.update({"class_probabilities":p,"predicted_class":pred,"confusion":cm,
            "class_report":pd.DataFrame({"class_name":STMTL_NAMES,"precision":precision,
                                         "recall":recall,"f1":f1,"support":cm.sum(1)})})
    if "segmentation" in values:
        q = np.asarray(values["segmentation"],np.float64)
        truth = np.asarray(masks)
        if q.shape != truth.shape or q.ndim != 4 or q.shape[0]!=n or q.shape[-1]!=1:
            raise ValueError("Invalid mask shape")
        if not np.isfinite(q).all() or ((q<0)|(q>1)).any() or not np.isin(truth,[0,1]).all():
            raise ValueError("Invalid mask probabilities or ground truth")
        t = truth[...,0]>0
        binary = q[...,0]>=.5
        ta,pa = t.sum((1,2)),binary.sum((1,2))
        positive,normal,malignant = ta>0,y==0,y==2
        if not np.array_equal(positive,y!=0):
            raise ValueError("Ground-truth mask presence disagrees with labels")
        if not positive.any() or not normal.any() or not malignant.any():
            raise ValueError("Missing validation subgroup")
        inter = (t&binary).sum((1,2))
        dice,iou = np.full(n,np.nan),np.full(n,np.nan)
        dice[positive] = 2*inter[positive]/(ta[positive]+pa[positive])
        iou[positive] = inter[positive]/(ta[positive]+pa[positive]-inter[positive])
        qa = q[...,0].sum((1,2))
        soft_inter = (q[...,0]*t).sum((1,2))
        soft_dice = (2*soft_inter+1e-6)/(ta+qa+1e-6)
        clipped = np.clip(q[...,0],1e-7,1-1e-7)
        bce = -(t*np.log(clipped)+(~t)*np.log1p(-clipped)).mean((1,2))
        seg_loss = bce+positive*(1-soft_dice)
        pixels = t.shape[1]*t.shape[2]
        stats.update({"val_positive_dice":float(dice[positive].mean()),
            "val_positive_iou":float(iou[positive].mean()),"val_malignant_dice":float(dice[malignant].mean()),
            "val_very_low_dice":int((dice[positive]<.1).sum()),
            "val_empty_positive_predictions":int((pa[positive]==0).sum()),
            "val_normal_with_any_foreground":int((pa[normal]>0).sum()),
            "val_normal_mean_foreground_percent":float(100*pa[normal].mean()/pixels),
            "val_normal_max_foreground_percent":float(100*pa[normal].max()/pixels),
            "val_segmentation_selection_loss":float(seg_loss.mean()),
            "val_positive_soft_dice":float(soft_dice[positive].mean())})
        detail.update({"dice":dice,"iou":iou,"predicted_pixels":pa,"true_pixels":ta,
                       "soft_dice":np.where(positive,soft_dice,np.nan),"pixel_bce":bce,
                       "segmentation_loss_per_image":seg_loss,
                       "mask_bits":np.packbits(binary.reshape(n,-1),axis=1),
                       "mask_shape":np.array(binary.shape[1:],np.int32)})
    if set(values)=={"classification","segmentation"}:
        pred,pa = detail["predicted_class"],detail["predicted_pixels"]
        a,b = int(((pred==0)&(pa>0)).sum()),int(((pred!=0)&(pa==0)).sum())
        stats.update({"val_inconsistent_pairs":a+b,"val_normal_class_with_mask":a,
                      "val_lesion_class_without_mask":b})
    if not stats or not all(np.isfinite(float(v)) for v in stats.values()):
        raise ValueError("Non-finite or empty validation statistics")
    return stats,detail


def _stmtl_selection(stats, task):
    if task=="classification":
        return (stats["val_malignant_balanced_accuracy"],stats["val_macro_f1"],
                -stats["val_negative_log_likelihood"])
    if task=="segmentation":
        return (stats["val_positive_dice"],-stats["val_segmentation_selection_loss"],
                -stats["val_normal_mean_foreground_percent"])
    raise ValueError("Unknown selection task")


def _stmtl_shared(network):
    return network.get_layer("shared_representation")


def _stmtl_assert_frozen(network, expected_digest):
    shared = _stmtl_shared(network)
    encoder = shared.get_layer("shared_encoder")
    if encoder.trainable or encoder.trainable_variables or not shared.trainable_variables:
        raise RuntimeError("CNN frozen / Transformer trainable assertion failed")
    if _phase_frozen_digest(shared,[]) != expected_digest:
        raise RuntimeError("Frozen CNN parameters or BatchNorm statistics changed")


def _stmtl_template(out, seed):
    directory = out/"initial_templates"
    directory.mkdir(exist_ok=True)
    path,meta = directory/f"seed{seed}.keras",directory/f"seed{seed}.json"
    if meta.is_file():
        record = json.loads(meta.read_text())
        if not path.is_file() or _repair_sha(path)!=record["file_sha256"]:
            raise ValueError("An initial template checkpoint changed")
        return path,record
    tf.keras.backend.clear_session()
    network = _stmtl_build_joint(seed)
    _save_model(network,path)
    record = {"seed":seed,"file_sha256":_repair_sha(path),
        "parameters":int(network.count_params()),
        "shared_sha256":_stmtl_weight_digest(_stmtl_shared(network)),
        "classification_head_sha256":_stmtl_weight_digest(network.get_layer("classification_head")),
        "segmentation_head_sha256":_stmtl_weight_digest(network.get_layer("segmentation_head")),
        "frozen_encoder_sha256":_phase_frozen_digest(_stmtl_shared(network),[]),
        "initialization":"ImageNet CNN only; fresh seeded Transformer and task heads; no BUSI-trained weights"}
    _write_json(meta,record)
    del network
    gc.collect()
    return path,record


def _stmtl_check_initial(network, mode, meta):
    _stmtl_assert_frozen(network,meta["frozen_encoder_sha256"])
    if _stmtl_weight_digest(_stmtl_shared(network)) != meta["shared_sha256"]:
        raise ValueError("Initial shared weights differ across modes")
    for task in STMTL_MODES[mode]:
        if _stmtl_weight_digest(network.get_layer(task+"_head"))!=meta[task+"_head_sha256"]:
            raise ValueError("Initial task-head weights differ across modes")


def _stmtl_smoke(template_path, meta, data, report):
    """Actual TensorFlow checks run in Colab, before any long training."""
    indices = [int(np.flatnonzero(data["y_train"]==i)[0]) for i in range(3)]
    x,y,m = data["X_train"][indices],data["y_train"][indices],data["M_train"][indices]
    records = []
    common = tf.keras.models.load_model(str(template_path),compile=False)
    reference = {k:v.numpy() for k,v in common(x,training=False).items()}
    for mode,active in STMTL_MODES.items():
        network = _stmtl_new_view(template_path,mode,42)
        _stmtl_check_initial(network,mode,meta)
        if {id(v) for v in network.weights}&{id(v) for v in common.weights}:
            raise RuntimeError("Independent models unexpectedly share variable objects")
        prediction = network(x,training=False)
        for task in active:
            np.testing.assert_allclose(prediction[task].numpy(),reference[task],atol=1e-6,rtol=1e-6)
        shared_variables = _stmtl_shared(network).trainable_variables
        norms = {}
        for task in active:
            with tf.GradientTape() as tape:
                pred = network(x,training=False)
                if task=="classification":
                    objective = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(y,pred[task]))
                else:
                    objective = tf.reduce_mean(segmentation_loss_per_image(m,pred[task]))
            gradients = tape.gradient(objective,shared_variables)
            connected = [g for g in gradients if g is not None]
            norm = float(tf.linalg.global_norm(connected).numpy()) if connected else 0.0
            if not np.isfinite(norm) or norm<=0:
                raise RuntimeError(task+" has no nonzero gradient into the shared Transformer")
            norms[task] = norm
        before = _stmtl_weight_digest(_stmtl_shared(network))
        target = _stmtl_targets(mode,y,m)
        network.reset_metrics()
        logs = network.train_on_batch(x,target,return_dict=True)
        if not all(np.isfinite(float(v)) for v in logs.values()):
            raise FloatingPointError("Non-finite smoke-test training metric")
        # Dropout is active in train_on_batch, so use a separate deterministic call
        # for the loss formula, rather than comparing training logs to inference.
        deterministic_losses = {}
        for task in active:
            if task=="classification":
                deterministic_losses[task] = float(tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
                    y,prediction[task])).numpy())
            else:
                observed_loss = segmentation_loss_per_image(m,prediction[task]).numpy()
                np_loss = _stmtl_statistics(y,m,{task:prediction[task].numpy()})[0]["val_segmentation_selection_loss"]
                np.testing.assert_allclose(float(observed_loss.mean()),np_loss,atol=1e-5,rtol=1e-5)
                deterministic_losses[task] = float(observed_loss.mean())
        compiled_loss = network.compute_loss(x=x,y=target,y_pred=prediction)
        np.testing.assert_allclose(float(compiled_loss.numpy()),sum(deterministic_losses.values()),atol=1e-5,rtol=1e-5)
        if _stmtl_weight_digest(_stmtl_shared(network))==before:
            raise RuntimeError("The shared Transformer did not update")
        _stmtl_assert_frozen(network,meta["frozen_encoder_sha256"])
        after = network(x,training=False)
        with tempfile.TemporaryDirectory(prefix="busi_stmtl_smoke_") as temp:
            path = Path(temp)/"model.keras"
            _save_model(network,path)
            restored = tf.keras.models.load_model(str(path))
            for task,value in restored(x,training=False).items():
                np.testing.assert_allclose(value.numpy(),after[task].numpy(),atol=1e-6,rtol=1e-6)
            if int(restored.optimizer.iterations.numpy())!=1:
                raise RuntimeError("Optimizer iteration did not survive serialization")
            original_optimizer = network.optimizer.variables
            restored_optimizer = restored.optimizer.variables
            if len(original_optimizer)!=len(restored_optimizer):
                raise RuntimeError("Optimizer slots were not restored")
            for a,b in zip(original_optimizer,restored_optimizer):
                np.testing.assert_allclose(a.numpy(),b.numpy(),atol=1e-7,rtol=1e-6)
            _stmtl_assert_frozen(restored,meta["frozen_encoder_sha256"])
            del restored
        records.append({"mode":mode,"active_tasks":list(active),"initial_weights_match":True,
            "unused_head_pruned":mode!="multitask","independent_model_variables":True,
            "shared_gradient_norm_by_task":norms,"shared_transformer_updated":True,
            "frozen_encoder_unchanged":True,"loss_formula_checked":True,
            "model_and_optimizer_reload":"passed","parameters":int(network.count_params()),
            "trainable_parameters":sum(int(np.prod(v.shape)) for v in network.trainable_variables)})
        del network,prediction,after,shared_variables
        gc.collect()
    del common,reference
    gc.collect()
    _write_json(report/"colab_smoke_test.json",records)
    print("All three real Colab smoke tests passed; training templates remain unchanged.",flush=True)


def _stmtl_record(folder, frame, data, values, mode, seed, selection_task, epoch, checkpoint_sha):
    folder.mkdir(parents=True,exist_ok=True)
    stats,detail = _stmtl_statistics(data["y_val"],data["M_val"],values)
    row = {"mode":mode,"seed":seed,"selection_task":selection_task,
           "selected_epoch":int(epoch),"initial_model_retained":int(epoch)==0,
           "checkpoint_sha256":checkpoint_sha,**stats}
    table = frame[[c for c in ["image","mask","class_name","group_id"] if c in frame]].copy()
    table["label"] = data["y_val"]
    arrays = {"images":frame["image"].to_numpy(dtype=str),"labels":data["y_val"]}
    if "classification" in values:
        p,pred = detail["class_probabilities"],detail["predicted_class"]
        arrays["classification"] = np.asarray(values["classification"])
        table["predicted_class"] = [STMTL_NAMES[i] for i in pred]
        table["classification_error"] = pred!=data["y_val"]
        for i,name in enumerate(STMTL_NAMES):table["p_"+name] = p[:,i]
        pd.DataFrame(detail["confusion"],index=STMTL_NAMES,columns=STMTL_NAMES).to_csv(folder/"confusion.csv")
        detail["class_report"].to_csv(folder/"classification_report.csv",index=False)
    if "segmentation" in values:
        arrays.update({key:detail[key] for key in ["mask_bits","mask_shape"]})
        for key in ["dice","iou","predicted_pixels","true_pixels","soft_dice","pixel_bce",
                    "segmentation_loss_per_image"]:table[key] = detail[key]
        table["predicted_foreground_percent"] = 100*detail["predicted_pixels"]/np.prod(detail["mask_shape"])
        grouped = table[table["label"]!=0].groupby("class_name",sort=False).agg(
            images=("image","count"),mean_dice=("dice","mean"),mean_iou=("iou","mean"))
        grouped.to_csv(folder/"segmentation_by_class.csv")
    np.savez_compressed(folder/"validation_predictions.npz",**arrays)
    table.to_csv(folder/"validation_per_image.csv",index=False)
    _write_json(folder/"summary.json",row)
    return row


def _stmtl_restore_state(directory, mode, template, seed, meta):
    state_path = directory/"state.json"
    if not state_path.is_file():
        return None,None
    state = json.loads(state_path.read_text())
    for task,item in state["best"].items():
        if task not in STMTL_MODES[mode] or _repair_sha(directory/item["file"])!=item["sha256"]:
            raise ValueError("Selected checkpoint state is invalid")
    epoch = int(state["completed_epochs"])
    if epoch==0:
        network = _stmtl_new_view(template,mode,seed)
        _stmtl_check_initial(network,mode,meta)
    else:
        path = directory/state["last_file"]
        if _repair_sha(path)!=state["last_sha256"]:
            raise ValueError("The committed optimizer checkpoint changed")
        network = tf.keras.models.load_model(str(path))
        if set(network.output)!=set(STMTL_MODES[mode]):
            raise ValueError("Restored tasks differ")
        expected_steps = epoch*math.ceil(499/STMTL_BATCH)
        if int(network.optimizer.iterations.numpy())!=expected_steps:
            raise ValueError("The optimizer checkpoint is not at the recorded epoch boundary")
        for i in range(1,epoch+1):
            if not (directory/"epochs"/f"{i:03d}.json").is_file():
                raise FileNotFoundError("A committed epoch report is missing")
    _stmtl_assert_frozen(network,meta["frozen_encoder_sha256"])
    return state,network


def _stmtl_run_one(out, report, template, meta, mode, seed, data, frame):
    directory = out/mode/f"seed{seed}"
    directory.mkdir(parents=True,exist_ok=True)
    (directory/"epochs").mkdir(exist_ok=True)
    _repair_guard(directory/"settings.json",{"study_settings_sha256":_repair_sha(out/"settings.json"),
        "seed":seed,"mode":mode,"initial_template_sha256":meta["file_sha256"]})
    if (directory/"completed.json").is_file():
        completed = json.loads((directory/"completed.json").read_text())
        for item in completed["selected"]:
            if _repair_sha(directory/item["file"])!=item["sha256"]:
                raise ValueError("A completed checkpoint changed")
        print("Reusing completed run:",mode,seed,flush=True)
        return [json.loads((report/item["report_folder"]/"summary.json").read_text())
                for item in completed["selected"]]
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    state,network = _stmtl_restore_state(directory,mode,template,seed,meta)
    if state is None:
        network = _stmtl_new_view(template,mode,seed)
        _stmtl_check_initial(network,mode,meta)
        initial_values = _stmtl_predict(network,data["X_val"])
        initial_stats = _stmtl_statistics(data["y_val"],data["M_val"],initial_values)[0]
        path = directory/"initial.keras"
        _save_model(network,path)
        sha = _repair_sha(path)
        state = {"completed_epochs":0,"last_file":None,"last_sha256":None,"best":{
            task:{"epoch":0,"file":path.name,"sha256":sha,"statistics":initial_stats,
                  "selection_tuple":list(_stmtl_selection(initial_stats,task))}
            for task in STMTL_MODES[mode]}}
        _write_json(directory/"initial_validation.json",initial_stats)
        _write_json(directory/"state.json",state)
        del initial_values
    _repair_guard(directory/"parameters.json",{"parameters":int(network.count_params()),
        "trainable_parameters":sum(int(np.prod(v.shape)) for v in network.trainable_variables),
        "shared_trainable_parameters":sum(int(np.prod(v.shape)) for v in _stmtl_shared(network).trainable_variables),
        "frozen_encoder_sha256":meta["frozen_encoder_sha256"]})
    steps = math.ceil(len(data["y_train"])/STMTL_BATCH)
    print("Training:",mode,"seed",seed,"from epoch",state["completed_epochs"],flush=True)
    for epoch in range(int(state["completed_epochs"]),STMTL_EPOCHS):
        x,m,y = _phase_epoch_arrays(data,seed,epoch)
        digest = hashlib.sha256()
        for array in [x,m,y]:digest.update(memoryview(np.ascontiguousarray(array)).cast("B"))
        fingerprint = out/"paired_epoch_inputs"/f"seed{seed}_epoch{epoch+1:02d}.json"
        fingerprint.parent.mkdir(exist_ok=True)
        _repair_guard(fingerprint,{"sha256":digest.hexdigest(),"seed":seed,"epoch_number":epoch+1})
        dataset = tf.data.Dataset.from_tensor_slices((x,_stmtl_targets(mode,y,m))).batch(STMTL_BATCH).prefetch(1)
        history = network.fit(dataset,initial_epoch=epoch,epochs=epoch+1,verbose=2)
        if int(network.optimizer.iterations.numpy())!=(epoch+1)*steps:
            raise RuntimeError("Unexpected optimizer step count")
        _stmtl_assert_frozen(network,meta["frozen_encoder_sha256"])
        values = _stmtl_predict(network,data["X_val"])
        stats = _stmtl_statistics(data["y_val"],data["M_val"],values)[0]
        improved = [task for task in STMTL_MODES[mode] if _stmtl_selection(stats,task)>
                    tuple(state["best"][task]["selection_tuple"])]
        # Double-buffer optimizer checkpoints. state.json commits an epoch only
        # after every referenced checkpoint/report has been written successfully.
        slot = "optimizer_b.keras" if state["last_file"]=="optimizer_a.keras" else "optimizer_a.keras"
        _save_model(network,directory/slot)
        next_state = json.loads(json.dumps(state))
        next_state.update({"completed_epochs":epoch+1,"last_file":slot,
                           "last_sha256":_repair_sha(directory/slot)})
        if improved:
            candidate = directory/f"candidate_{epoch+1:03d}.keras"
            shutil.copyfile(directory/slot,candidate)
            sha = _repair_sha(candidate)
            for task in improved:
                next_state["best"][task] = {"epoch":epoch+1,"file":candidate.name,"sha256":sha,
                    "statistics":stats,"selection_tuple":list(_stmtl_selection(stats,task))}
        log = {"epoch":epoch+1,**{k:float(v[-1]) for k,v in history.history.items()},**stats}
        _write_json(directory/"epochs"/f"{epoch+1:03d}.json",log)
        _write_json(directory/"state.json",next_state)
        state = next_state
        # Only this study's obsolete, unreferenced candidate files are removed.
        keep = {item["file"] for item in state["best"].values()}
        for path in directory.glob("candidate_*.keras"):
            if path.name not in keep:path.unlink()
        keys = ["val_malignant_balanced_accuracy","val_positive_dice"]
        print("Validation:",{k:round(stats[k],6) for k in keys if k in stats},
              "| selected:",{k:v["epoch"] for k,v in state["best"].items()},flush=True)
        del dataset,x,m,y,values
        gc.collect()
    records = [json.loads((directory/"epochs"/f"{i:03d}.json").read_text())
               for i in range(1,int(state["completed_epochs"])+1)]
    pd.DataFrame(records).to_csv(directory/"training_log.csv",index=False)
    del network
    gc.collect()
    rows,selected = [],[]
    params = json.loads((directory/"parameters.json").read_text())
    for task,item in state["best"].items():
        path = directory/item["file"]
        if _repair_sha(path)!=item["sha256"]:raise ValueError("Selected checkpoint hash mismatch")
        network = tf.keras.models.load_model(str(path),compile=False)
        _stmtl_assert_frozen(network,meta["frozen_encoder_sha256"])
        values = _stmtl_predict(network,data["X_val"])
        stats = _stmtl_statistics(data["y_val"],data["M_val"],values)[0]
        for key,value in item["statistics"].items():
            np.testing.assert_allclose(stats[key],value,atol=1e-6,rtol=1e-6,
                                       err_msg="Reloaded checkpoint validation differs")
        folder_name = f"{mode}_seed{seed}_selected_{task}"
        destination = report/folder_name
        row = _stmtl_record(destination,frame,data,values,mode,seed,task,item["epoch"],item["sha256"])
        row.update(params)
        _write_json(destination/"summary.json",row)
        for filename in ["settings.json","parameters.json","training_log.csv","initial_validation.json","state.json"]:
            shutil.copyfile(directory/filename,destination/filename)
        rows.append(row)
        selected.append({"file":item["file"],"sha256":item["sha256"],"selection_task":task,
                         "report_folder":folder_name})
        del network,values
        gc.collect()
    _write_json(directory/"completed.json",{"completed_epochs":STMTL_EPOCHS,"selected":selected,"test_evaluated":False})
    return rows


def _stmtl_tables(report, rows):
    full = pd.DataFrame(rows)
    if len(full)!=12:raise ValueError("Expected 3 ST-class, 3 ST-seg and 6 task-selected MTL records")
    full.to_csv(report/"all_selected_checkpoints.csv",index=False)
    fields = {
        "classification":["val_accuracy","val_macro_f1","val_malignant_recall","val_missed_malignant",
            "val_nonmalignant_predicted_malignant","val_nonmalignant_specificity","val_malignant_balanced_accuracy",
            "val_normal_recall","val_malignant_auc","val_macro_auc_ovr","val_ece_15bins",
            "val_brier_multiclass_sum","val_negative_log_likelihood"],
        "segmentation":["val_positive_dice","val_positive_iou","val_malignant_dice","val_very_low_dice",
            "val_empty_positive_predictions","val_normal_with_any_foreground","val_normal_mean_foreground_percent",
            "val_normal_max_foreground_percent","val_segmentation_selection_loss","val_positive_soft_dice"]}
    tables = {}
    effects = {}
    for task,columns in fields.items():
        subset = full[full["selection_task"]==task]
        table = subset[["mode","seed","selection_task","selected_epoch","initial_model_retained",
                        "parameters","trainable_parameters"]+columns].copy()
        if len(table)!=6 or table[columns].isna().any().any():
            raise ValueError("Missing task-specific comparison metric")
        table.to_csv(report/f"{task}_comparison.csv",index=False)
        table.groupby("mode",sort=False)[columns].agg(["mean","std"]).to_csv(report/f"{task}_mean_std.csv")
        single = table[table["mode"]==task+"_only"].set_index("seed")
        multi = table[table["mode"]=="multitask"].set_index("seed")
        if set(single.index)!=set(STMTL_SEEDS) or set(multi.index)!=set(STMTL_SEEDS):
            raise ValueError("Unpaired study seeds")
        difference = multi[columns]-single[columns]
        difference.to_csv(report/f"{task}_paired_MTL_minus_ST.csv")
        effects[task] = {"mean_MTL_minus_ST":{k:float(v) for k,v in difference.mean().items()},
            "comparison":"same seed; each method selected by the same criterion for this task"}
        tables[task] = table
    parameter_table = full.drop_duplicates(["mode","seed"])[["mode","seed","parameters","trainable_parameters",
                                                               "shared_trainable_parameters"]]
    parameter_table.to_csv(report/"parameter_counts.csv",index=False)
    _write_json(report/"comparison_notes.json",{"effects":effects,"test_evaluated":False,
        "MTL_checkpoints":"Classification and segmentation are selected separately. Do not combine their scores as one checkpoint.",
        "statistics":"Three paired seeds are descriptive repeats on one fixed validation split, not independent datasets.",
        "scope":"Fresh ImageNet-initialized CNN+Transformer core with independent heads; not the full LPRM/adaptive-weight model.",
        "parameter_counts":"Architecture counts only; two task-selected MTL checkpoints are distinct deployment states. No latency/memory claim.",
        "test_history":"Earlier test results have already been inspected; this is exploratory validation evidence."})
    return tables


def run_busi_single_vs_multi(experiments_root="/content/drive/MyDrive/BUSI_EXPERIMENTS",
                             dataset_root="/content/drive/MyDrive/BUSI_MERGED",evaluation_dir=None):
    evaluation = _repair_choose_protocol(experiments_root,evaluation_dir)
    protocol_file = evaluation/"frozen_protocol.json"
    protocol = json.loads(protocol_file.read_text())
    completed = json.loads((evaluation/"completed.json").read_text())
    if completed["protocol_sha256"]!=_repair_sha(protocol_file):
        raise ValueError("The original frozen protocol changed")
    if protocol["class_order"]!=STMTL_NAMES or protocol["mask_threshold"]!=.5:
        raise ValueError("Unexpected class order or threshold")
    anchors = [item for item in protocol["checkpoints"] if item["variant"]=="full_lprm_v1"]
    sources = {str(Path(item["model"]).parent.parent) for item in anchors}
    if len(sources)!=1:raise ValueError("Cannot locate the source manifest/configuration")
    source = Path(next(iter(sources)))
    manifest_file = source/"split_manifest.csv"
    if _repair_sha(manifest_file)!=protocol["manifest_sha256"]:
        raise ValueError("The original split manifest changed")
    manifest = pd.read_csv(manifest_file)
    if len(manifest)!=761 or not manifest["image"].is_unique or manifest["split"].value_counts().to_dict()!={"train":499,"val":115,"test":147}:
        raise ValueError("Unexpected fixed split")
    if manifest["group_id"].isna().any() or manifest.groupby("group_id")["split"].nunique().max()!=1:
        raise ValueError("Similarity groups cross split boundaries")
    for column in ["image","mask"]:
        if any(Path(p).is_absolute() or ".." in Path(p).parts for p in manifest[column]):
            raise ValueError("Expected relative data paths")
    normalization = json.loads((source/"source_config.json").read_text())["normalization"]
    for key in ["mean","std"]:
        np.testing.assert_allclose(normalization[key],protocol["normalization"][key],atol=1e-7,rtol=0)
    out = evaluation.parent/"single_vs_multi_v1"
    out.mkdir(exist_ok=True)
    report = out/"reports"
    report.mkdir(exist_ok=True)
    settings = {"version":STMTL_VERSION,"seeds":STMTL_SEEDS,"modes":{k:list(v) for k,v in STMTL_MODES.items()},
        "fixed_epochs_per_run":STMTL_EPOCHS,"early_stopping":False,"runs":9,"batch_size":STMTL_BATCH,
        "optimizer":"fresh Adam","learning_rate":1e-4,"clipnorm":1.0,
        "initialization":"CNN ImageNet only; freshly seeded Transformer/heads; common per-seed template cloned independently",
        "no_previously_BUSI_trained_weights":True,"CNN":"MobileNetV2 alpha=1, frozen including BN",
        "Transformer":"shared 49x128 tokens; four heads; FFN 256; zero-initialized residual projection",
        "heads":"independent GAP+Dense classifier and attention-gated segmentation decoder; unused head physically pruned in ST",
        "LPRM":False,"adaptive_task_weights":False,"class_cost_weights":[1,1,1],"task_loss_weights":[1,1],
        "classification_loss":"mean plain CE","segmentation_loss":"mean BCE_all + Dice_on_positive_images",
        "preprocessing":"PIL RGB bilinear 224 uint8 -> float32 raw pixels; model rescales to [-1,1] for MobileNetV2",
        "old_normalization":"saved train Z-score is undone before this study; not applied twice",
        "augmentation":"identical stateless epoch shuffle and paired horizontal flips p=.5 across modes",
        "classification_selection":["max malignant BA","max macro F1","min NLL"],
        "segmentation_selection":["max positive hard Dice","min BCE+positive soft-Dice validation loss","min normal foreground area"],
        "MTL_selection":"two separate task-selected checkpoints; no splicing their metrics into one model",
        "initial_checkpoint_included":True,"mask_threshold":.5,"class_prediction":"argmax",
        "manifest_sha256":_repair_sha(manifest_file),"source_config_sha256":_repair_sha(source/"source_config.json"),
        "frozen_protocol_sha256":_repair_sha(protocol_file),"source_directory":str(source),
        "test_evaluated":False,"similarity_groups":"no verified patient-ID claim",
        "interpretation":"controlled core comparison with unit loss weights, not a re-estimate of historical full-model performance",
        "test_history":"test has already been inspected in earlier experiments; this is exploratory validation only",
        "resume":"committed epoch and optimizer resume; dropout replay after interruption is not bit exact",
        "versions":{"tensorflow":tf.__version__,"keras":tf.keras.__version__,"numpy":np.__version__,"python":platform.python_version()}}
    _repair_guard(out/"settings.json",settings)
    shutil.copyfile(out/"settings.json",report/"settings.json")
    shutil.copyfile(protocol_file,report/"source_frozen_protocol.json")
    if (out/"completed.json").is_file():
        saved = json.loads((out/"completed.json").read_text())
        for filename,sha in saved["report_sha256"].items():
            if _repair_sha(report/filename)!=sha:raise ValueError("A completed study report changed")
        archive = shutil.make_archive("/content/BUSI_Single_vs_Multi_RESULTS","zip",root_dir=report)
        return {"directory":out,"classification":pd.read_csv(report/"classification_comparison.csv"),
                "segmentation":pd.read_csv(report/"segmentation_comparison.csv"),"zip_path":archive}
    data,frames = _read_data(source,manifest,Path(dataset_root),None)
    mean,std = np.array(normalization["mean"],np.float32),np.array(normalization["std"],np.float32)
    for split in ["train","val"]:
        # _read_data resized uint8 RGB then Z-scored; reverse exactly once.
        data["X_"+split] = np.clip(np.rint(data["X_"+split]*std+mean),0,255).astype(np.float32)
    digest = hashlib.sha256()
    for key in ["X_train","M_train","y_train","X_val","M_val","y_val"]:
        array = np.ascontiguousarray(data[key])
        digest.update(str((key,array.shape,array.dtype.str)).encode())
        digest.update(memoryview(array).cast("B"))
    _repair_guard(out/"input_arrays.json",{"sha256":digest.hexdigest(),"train":499,"validation":115,"test_loaded":False})
    shutil.copyfile(out/"input_arrays.json",report/"input_arrays.json")
    frames["val"].to_csv(report/"validation_manifest.csv",index=False)
    np.savez_compressed(report/"validation_reference.npz",images=frames["val"]["image"].to_numpy(dtype=str),
        labels=data["y_val"],mask_bits=np.packbits((data["M_val"][...,0]>0).reshape(115,-1),axis=1),
        mask_shape=np.array([224,224],np.int32))
    template,meta = _stmtl_template(out,42)
    _stmtl_smoke(template,meta,data,report)
    rows = []
    for seed in STMTL_SEEDS:
        template,meta = _stmtl_template(out,seed)
        _repair_guard(out/"common_imagenet_encoder.json",{"sha256":meta["frozen_encoder_sha256"]})
        shutil.copyfile(template.with_suffix(".json"),report/f"initial_template_seed{seed}.json")
        for mode in STMTL_MODES:
            rows.extend(_stmtl_run_one(out,report,template,meta,mode,seed,data,frames["val"]))
        pd.DataFrame(rows).to_csv(report/"progress_selected_checkpoints.csv",index=False)
    tables = _stmtl_tables(report,rows)
    fingerprints = out/"paired_epoch_inputs"
    shutil.copytree(fingerprints,report/"paired_epoch_inputs",dirs_exist_ok=True)
    instructions = (
        "BUSI single-task versus multi-task CORE study (validation only).\n"
        "Nine fresh trainings: 3 modes x seeds 42,43,44, each 30 epochs, batch 8, Adam 1e-4.\n"
        "Only the CNN is ImageNet-pretrained; no BUSI-trained weight warm start.\n"
        "Shared CNN frozen; shared Transformer trainable; same head initialization for matched tasks.\n"
        "LPRM, adaptive task weights and class costs disabled here to isolate core task sharing.\n"
        "Classification-only trains without mask targets; segmentation-only trains without class targets.\n"
        "Same epoch order/flips/initial weights, independently loaded models, fixed split and mask threshold.\n"
        "MTL classification and segmentation checkpoints are selected separately by task-matched criteria.\n"
        "Do not combine scores from two different selected checkpoints as a single model result.\n"
        "all_selected_checkpoints.csv retains both real outputs for each MTL checkpoint.\n"
        "Exact thresholded masks are packed in NPZ; per-image soft Dice/BCE are in CSV. Soft mask maps can be regenerated from saved models.\n"
        "Classification labels are used for segmentation subgroup evaluation only, not segmentation training.\n"
        "Parameters describe architecture size, not latency or deployment memory.\n"
        "Previous test results have already been inspected; this follow-up provides exploratory validation evidence.\n"
        "No test images are loaded. Similarity groups are not verified patient IDs.\n"
        "ImageNet preprocessing: https://keras.io/api/applications/mobilenet/\n"
        "Functional task graphs: https://keras.io/guides/functional_api/\n"
    )
    (report/"README.txt").write_text(instructions,encoding="utf-8")
    report_sha = {str(path.relative_to(report)):_repair_sha(path) for path in sorted(report.rglob("*")) if path.is_file()}
    _write_json(out/"completed.json",{"report_sha256":report_sha,"test_evaluated":False,"runs":9})
    archive = shutil.make_archive("/content/BUSI_Single_vs_Multi_RESULTS","zip",root_dir=report)
    print("Finished: 9 trainings; task-matched validation tables saved. No test images loaded.",flush=True)
    return {"directory":out,"classification":tables["classification"],"segmentation":tables["segmentation"],"zip_path":archive}


In [ ]:
#@title ۳. اجرای مقایسه و دانلود گزارش
from IPython.display import display
from google.colab import files

single_multi_result = run_busi_single_vs_multi()

print("CLASSIFICATION — checkpoints selected for classification")
display(single_multi_result["classification"][[
    "mode", "seed", "selected_epoch", "val_accuracy", "val_macro_f1",
    "val_malignant_recall", "val_missed_malignant",
    "val_nonmalignant_predicted_malignant", "val_malignant_balanced_accuracy"
]])
print("SEGMENTATION — checkpoints selected for segmentation")
display(single_multi_result["segmentation"][[
    "mode", "seed", "selected_epoch", "val_positive_dice", "val_positive_iou",
    "val_malignant_dice", "val_empty_positive_predictions",
    "val_normal_mean_foreground_percent"
]])
print("Saved:", single_multi_result["directory"])
files.download(single_multi_result["zip_path"])


<div dir="rtl" align="right">
<p>پیش از آموزش، خود Colab یکسان بودن وزن‌های شروع، حذف سرِ وظیفهٔ غیرفعال، غیرصفر بودن گرادیان هر وظیفه در Transformer، ثابت ماندن CNN و ذخیره/بارگذاری مدل و optimizer را کنترل می‌کند. این آموزش‌های کوتاهِ کنترلی روی مدل‌های دورریختنی‌اند و وزن شروع آزمایش‌ها را تغییر نمی‌دهند.</p>
<p>برای طبقه‌بندی، checkpoint بر اساس malignant balanced accuracy انتخاب می‌شود؛ در تساوی، macro F1 و سپس NLL تعیین‌کننده‌اند. برای سگمنتیشن، ابتدا positive Dice و در تساوی loss سگمنتیشن و سپس مساحت مثبت کاذب در تصاویر normal بررسی می‌شود. مدل اولیهٔ epoch صفر هم کاندیداست. هیچ early stopping استفاده نمی‌شود تا بودجهٔ همهٔ حالت‌ها ۳۰ epoch باشد.</p>
<p>مدل مشترک دو checkpoint منتخب خواهد داشت: یکی برای مقایسهٔ طبقه‌بندی و یکی برای مقایسهٔ سگمنتیشن. امتیازهای این دو checkpoint را نباید به‌عنوان نتیجهٔ یک مدل واحد کنار هم گزارش کرد. فایل <code dir="ltr">all_selected_checkpoints.csv</code> خروجی هر checkpoint را جدا ثبت می‌کند.</p>
<p>در پایان، <code dir="ltr">BUSI_Single_vs_Multi_RESULTS.zip</code> دانلود می‌شود؛ آن را برای بررسی بفرست. مدل‌ها داخل پوشهٔ <code dir="ltr">single_vs_multi_v1</code> در پروژهٔ قبلی ذخیره می‌شوند. اگر اجرا قطع شد، همین نوت‌بوک را دوباره اجرا کن؛ از آخرین epoch ثبت‌شده ادامه می‌دهد. مسیر تصادفی dropout پس از قطع اجرا الزاماً عین اجرای پیوسته نیست.</p>
<p>جدول‌ها mean و std روی سه seed و اختلاف زوجی MTL با ST را ذخیره می‌کنند. این سه تکرار، سه دیتاست مستقل نیستند و گروه‌های شباهت هم شناسهٔ تأییدشدهٔ بیمار نیستند. با توجه به مشاهدهٔ test در مراحل قبلی، این مرحله شاهد اکتشافی روی validation است.</p>
<p>کد و محاسبات مستقل از TensorFlow به‌صورت محلی بررسی شده‌اند؛ اجرای واقعی مدل، گرادیان و ذخیره/بارگذاری در ابتدای اجرای همین نوت‌بوک در Colab بررسی می‌شود.</p>
</div>
